In [ ]:
import pandas as pd
import psycopg2

df_group = pd.read_csv(r'C:\Users\Administrator\Desktop\data_learn\eCommerce_Events_History\data\interim\03_user_behavior_groups.csv')
filter_columns = ['user_session', 'product_id', 'price', 'user_id', 'group_type']
df_group = df_group[filter_columns]


conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user='postgres',
    password='',
    port=5432
)

cur = conn.cursor()

query = """
SELECT 
    user_session,
    product_id,
    user_id,
    brand,
FROM makeup_consumer_events.dec
WHERE event_type IN ('cart', 'purchase', 'remove_from_cart')
ORDER BY user_session, event_time
"""

cur.execute(query)
results = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_events = pd.DataFrame(results, columns=columns)

cur.close()
conn.close()

# 给abc组加上品牌字段
df = pd.merge(df_group, df_events, on=['user_session', 'product_id', 'user_id'], how='left')
df = df.drop_duplicates(subset=['user_session', 'product_id', 'user_id'])
display(df.head())


,user_session,product_id,price,user_id,group_type,brand
0,00002b0e-d7f7-454e-8386-431c4021a9f6,4905,1.19,531784651,C,runail
2,00002b0e-d7f7-454e-8386-431c4021a9f6,5622678,9.11,531784651,C,severina
4,00002b0e-d7f7-454e-8386-431c4021a9f6,5622689,2.14,531784651,C,NaN
6,00002b0e-d7f7-454e-8386-431c4021a9f6,5645160,0.49,531784651,B,NaN
15,00002b0e-d7f7-454e-8386-431c4021a9f6,5650591,4.76,531784651,C,metzger


In [26]:
# 1. 检查 brand 列的实际数据类型和缺失值情况
# print(f"df['brand'].dtype: {df['brand'].dtype}")
# print(f"df['brand'].isnull().sum(): {df['brand'].isnull().sum()}")
# print(f"df['brand'].isna().sum(): {df['brand'].isna().sum()}")

df['brand'] = df['brand'].replace('NaN', 'Unknown Brand')
# display(df.head())

# 行设置为 brand（品牌），列设置为 group_type（A/B/C组）
df_group_brand = df.groupby('brand')['group_type'].value_counts().unstack(fill_value=0)

# axis=0 这里是用来求出每列（每个组）的总量，然后再做除法
df_brand_pct = df_group_brand.div(df_group_brand.sum(axis=0), axis=1) * 100

# 过滤掉全为0的行（实际上是极小值被round成0的）
df_brand_pct = df_brand_pct[(df_brand_pct > 0.005).any(axis=1 )]

# 现在列名是 A, B, C了，所以可以按 A组 从高到低排序
df_brand_pct = df_brand_pct.sort_values(by='A', ascending=False)
df_brand_pct = df_brand_pct.round(2)

# 品牌太多，展示前20名即可
display(df_brand_pct.head(20))


group_type,A,B,C
brand,,,
Unknown Brand,43.01,44.19,44.48
runail,8.54,8.18,7.39
irisk,4.96,4.77,4.44
grattol,3.84,4.47,4.10
bpw.style,3.29,2.81,2.73
masura,3.26,3.93,5.02
ingarden,1.95,2.07,2.24
estel,1.93,1.65,1.51
kapous,1.50,1.31,1.12


In [29]:
# 找出 B 组和 C 组偏好明显高于 A 组的品牌

# 把 Unknown Brand 排除，避免它的巨大占比干扰核心品牌的比较
df_real_brand = df_brand_pct.drop('Unknown Brand', errors='ignore')

# 分别计算 B-A 和 C-A 的百分比差值（B和C比A高出多少特征）
df_real_brand['A_minus_B'] = df_real_brand['A'] - df_real_brand['B']
df_real_brand['A_minus_C'] = df_real_brand['A'] - df_real_brand['C']

# 查看哪些品牌 A 组的购买倾向明显高于 B/C 组
a_prefer_brands = df_real_brand.sort_values(by='A_minus_C', ascending=False)
display(a_prefer_brands[['A', 'B', 'C', 'A_minus_B', 'A_minus_C']])


group_type,A,B,C,A_minus_B,A_minus_C
brand,,,,,
runail,8.54,8.18,7.39,0.36,1.15
bpw.style,3.29,2.81,2.73,0.48,0.56
irisk,4.96,4.77,4.44,0.19,0.52
italwax,1.18,0.90,0.74,0.28,0.44
estel,1.93,1.65,1.51,0.28,0.42
...,...,...,...,...,...
oniq,0.33,0.46,0.62,-0.13,-0.29
ingarden,1.95,2.07,2.24,-0.12,-0.29
haruyama,0.82,0.95,1.15,-0.13,-0.33
